# Notebook 8.3 -- Double Descent

This notebook investigates double descent as described in section 8.4 of the book.

It uses the MNIST-1D database which can be found at https://github.com/greydanus/mnist1d

Work through the cells below, running each cell in turn. In various places you will see the words "TO DO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

In [ ]:
# Run this once to install the MNIST-1D package (on Colab or on your own machine)
%pip install mnist1d

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import numpy as np
import matplotlib.pyplot as plt
import mnist1d
import random
random.seed(0)
torch.manual_seed(0)

# Try attaching to GPU -- on Colab, use "Runtime > Change runtime type" to select a GPU
DEVICE = str(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
print('Using:', DEVICE)

In [ ]:
args = mnist1d.data.get_dataset_args()
args.num_samples = 8000
args.train_split = 0.5
args.corr_noise_scale = 0.25
args.iid_noise_scale=2e-2
data = mnist1d.data.get_dataset(args, path='./mnist1d_data_double_descent.pkl', download=False, regenerate=True)

# Add 15% noise to training labels
for c_y in range(len(data['y'])):
    random_number = random.random()
    if random_number < 0.15 :
        random_int = int(random.random() * 10)
        data['y'][c_y] = random_int

# The training and test input and outputs are in
# data['x'], data['y'], data['x_test'], and data['y_test']
print("Examples in training set: {}".format(len(data['y'])))
print("Examples in test set: {}".format(len(data['y_test'])))
print("Length of each example: {}".format(data['x'].shape[-1]))

In [ ]:
# Initialize the parameters with He initialization
def weights_init(layer_in):
  if isinstance(layer_in, nn.Linear):
    nn.init.kaiming_uniform_(layer_in.weight)
    layer_in.bias.data.fill_(0.0)

# Return an initialized model with two hidden layers and n_hidden hidden units at each
def get_model(n_hidden):

  D_i = 40    # Input dimensions
  D_k = n_hidden   # Hidden dimensions
  D_o = 10    # Output dimensions

  # Define a model with two hidden layers of size n_hidden
  # And ReLU activations between them
  model = nn.Sequential(
  nn.Linear(D_i, D_k),
  nn.ReLU(),
  nn.Linear(D_k, D_k),
  nn.ReLU(),
  nn.Linear(D_k, D_o))

  # Initialize the weights
  model.apply(weights_init)

  # Return the model
  return model ;

# Count the trainable parameters (weights and biases) of a model
def count_parameters(model):
  return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def fit_model(model, data):

  # Move the model to the GPU if there is one
  model = model.to(DEVICE)

  # choose cross entropy loss function (equation 5.24)
  loss_function = torch.nn.CrossEntropyLoss()
  # construct SGD optimizer and initialize learning rate and momentum
  # optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
  optimizer = torch.optim.SGD(model.parameters(), lr = 0.01, momentum=0.9)


  # convert the data to PyTorch tensors on the same device as the model
  x_train = torch.tensor(data['x'].astype('float32')).to(DEVICE)
  y_train = torch.tensor(data['y'].transpose().astype('int64')).to(DEVICE)
  x_test= torch.tensor(data['x_test'].astype('float32')).to(DEVICE)
  y_test = torch.tensor(data['y_test'].astype('int64')).to(DEVICE)

  # load the data into a class that creates the batches (seeded so the shuffling is repeatable)
  torch.manual_seed(1)
  data_loader = DataLoader(TensorDataset(x_train,y_train), batch_size=100, shuffle=True)

  # loop over the dataset n_epoch times
  n_epoch = 1000

  for epoch in range(n_epoch):
    # loop over batches
    for i, batch in enumerate(data_loader):
      # retrieve inputs and labels for this batch
      x_batch, y_batch = batch
      # zero the parameter gradients
      optimizer.zero_grad()
      # forward pass -- calculate model output
      pred = model(x_batch)
      # compute the loss
      loss = loss_function(pred, y_batch)
      # backward pass
      loss.backward()
      # SGD update
      optimizer.step()

    # Run whole dataset to get statistics -- normally wouldn't do this
    with torch.no_grad():
      pred_train = model(x_train)
      pred_test = model(x_test)
    _, predicted_train_class = torch.max(pred_train.data, 1)
    _, predicted_test_class = torch.max(pred_test.data, 1)
    errors_train = 100 - 100 * (predicted_train_class == y_train).float().sum().item() / len(y_train)
    errors_test= 100 - 100 * (predicted_test_class == y_test).float().sum().item() / len(y_test)
    losses_train = loss_function(pred_train, y_train).item()
    losses_test= loss_function(pred_test, y_test).item()
    if epoch%100 ==0 :
      print(f'Epoch {epoch:5d}, train loss {losses_train:.6f}, train error {errors_train:3.2f},  test loss {losses_test:.6f}, test error {errors_test:3.2f}')

  return errors_train, errors_test


The following code produces the double descent curve by training the model with different numbers of hidden units and plotting the test error.

TO DO:

*Before* you run the code, and considering that there are 4000 training examples predict:<br>

1.    At what capacity do you think the training error will become zero?
2.   At what capacity do you expect the first minimum of the double descent curve to appear?
3. At what capacity do you expect the maximum of the double descent curve to appear?

Hint: `count_parameters(get_model(n_hidden))` tells you how many parameters a model with `n_hidden` hidden units per layer has.

In [ ]:
# This code will take a while (~30 mins on GPU) to run!  Go and make a cup of coffee!

hidden_variables = np.array([2,4,6,8,10,14,18,22,26,30,35,40,45,50,55,60,70,80,90,100,120,140,160,180,200,250,300,400]) ;
# Use float arrays: np.zeros_like(hidden_variables) would be an integer array and truncate the error percentages
errors_train_all = np.zeros(len(hidden_variables))
errors_test_all = np.zeros(len(hidden_variables))
n_params_all = np.zeros(len(hidden_variables), dtype=int)

# For each hidden variable size
for c_hidden in range(len(hidden_variables)):
    print(f'Training model with {hidden_variables[c_hidden]:3d} hidden variables')
    # Get a model
    model = get_model(hidden_variables[c_hidden]) ;
    n_params_all[c_hidden] = count_parameters(model)
    # Train the model
    errors_train, errors_test = fit_model(model, data)
    # Store the results
    errors_train_all[c_hidden] = errors_train
    errors_test_all[c_hidden]= errors_test

In [ ]:
# Plot the results
fig, ax = plt.subplots()
ax.plot(hidden_variables, errors_train_all,'r-',label='train')
ax.plot(hidden_variables, errors_test_all,'b-',label='test')
# Mark the capacity where the number of parameters is closest to the number of training examples
n_train = len(data['y'])
hidden_at_n_train = hidden_variables[np.argmin(np.abs(n_params_all - n_train))]
ax.axvline(x=hidden_at_n_train, color='g', linestyle='--', label='parameters = training examples')
ax.set_ylim(0,100);
ax.set_xlabel('Number of hidden units per layer'); ax.set_ylabel('Error (%)')
ax.legend()
plt.show()
